# Recommendation System Evaluation

This notebook evaluates ALS, LSH, and Hybrid recommenders with timing measurements.

In [1]:
import sys
import time
sys.path.append("..")

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, min as spark_min, max as spark_max

from src import config
from src.data_loader import load_ratings, load_movies, load_embeddings, load_enriched_movies
from src.features import build_movie_features, build_user_features
from src.lsh_recommender import LSHRecommender
from src.als_recommender import ALSRecommender
from src.hybrid_recommender import HybridRecommender
from src.evaluation import evaluate_recommendations

## Initialize Spark Session

In [2]:
spark = SparkSession.builder \
    .appName("MMDS") \
    .master("local[*]") \
    .config("spark.local.dir", config.SPARK_TMP_DIR) \
    .config("spark.driver.memory", config.DRIVER_MEMORY) \
    .config("spark.executor.memory", config.EXECUTOR_MEMORY) \
    .getOrCreate()

# Store timing results
timing_results = {}

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/18 18:35:47 WARN Utils: Your hostname, MacBook-Pro-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.145 instead (on interface en0)
26/01/18 18:35:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/18 18:35:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/18 18:35:47 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).
26/01/18 18:35:48 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/01/18 18:35:48 WARN Utils: S

## Load Data

In [3]:
train_ratings, test_ratings = load_ratings(spark)
movies = load_movies(spark)
embeddings_df = load_embeddings(spark)
movies_enriched = load_enriched_movies(spark)

print(f"Train ratings: {train_ratings.count()}")
print(f"Test ratings: {test_ratings.count()}")

Train ratings: 802553
Test ratings: 197656


## Test Set Statistics

In [4]:
test_stats = test_ratings.groupBy("user_id").count()
test_stats.describe("count").show()

print(f"\nTotal test users: {test_ratings.select('user_id').distinct().count()}")
print(f"Total test ratings: {test_ratings.count()}")

print(f"\n=== Relevant Items Per User (rating >= {config.RELEVANCE_THRESHOLD}) ===")
relevant_per_user = (
    test_ratings
    .filter(col("rating") >= config.RELEVANCE_THRESHOLD)
    .groupBy("user_id")
    .agg(count("*").alias("relevant_count"))
)

relevant_stats = relevant_per_user.agg(
    avg("relevant_count").alias("avg_relevant"),
    spark_min("relevant_count").alias("min_relevant"),
    spark_max("relevant_count").alias("max_relevant")
)
relevant_stats.show()

users_with_few_relevant = relevant_per_user.filter(col("relevant_count") < config.TOP_K_PRECISION).count()
total_users = relevant_per_user.count()
print(f"Users with < {config.TOP_K_PRECISION} relevant items: {users_with_few_relevant} / {total_users} ({100.0 * users_with_few_relevant / total_users:.1f}%)")

+-------+------------------+
|summary|             count|
+-------+------------------+
|  count|              6040|
|   mean|32.724503311258275|
| stddev|38.540047350705976|
|    min|                 4|
|    max|               462|
+-------+------------------+


Total test users: 6040
Total test ratings: 197656

=== Relevant Items Per User (rating >= 4) ===
+------------------+------------+------------+
|      avg_relevant|min_relevant|max_relevant|
+------------------+------------+------------+
|17.307330984734104|           1|         229|
+------------------+------------+------------+

Users with < 10 relevant items: 2754 / 5961 (46.2%)


## Build Features

In [5]:
movies_profiles = build_movie_features(movies, embeddings_df, movies_enriched)
user_profiles = build_user_features(train_ratings, movies_profiles)

test_users = test_ratings.select("user_id").distinct()
test_user_profiles = user_profiles.join(test_users, on="user_id", how="inner")

print(f"Movie profiles: {movies_profiles.count()}")
print(f"User profiles: {user_profiles.count()}")
print(f"Test user profiles: {test_user_profiles.count()}")

Movie profiles: 3883
User profiles: 6040
Test user profiles: 6040


## ALS Recommender

In [6]:
als_rec = ALSRecommender()

als_rec, als_train_time = als_rec.fit(train_ratings)
timing_results["ALS"] = {"train": als_train_time}
print(f"ALS Training Time: {als_train_time:.2f}s")

als_recs, als_inference_time = als_rec.recommend(test_users, train_ratings)
timing_results["ALS"]["inference"] = als_inference_time
print(f"ALS Inference Time: {als_inference_time:.2f}s")

26/01/18 18:35:57 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


ALS Training Time: 6.09s


ALS Inference Time: 12.39s


In [7]:
als_metrics = evaluate_recommendations(als_recs, test_ratings)
print("ALS Metrics:")
als_metrics.show()

/Users/oleksandr/Development/Study/Ucu/MMD/part2/.venv/lib/python3.12/site-packages/pyspark/sql/context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


ALS Metrics:
+-------------------+------------------+-------------------+
|   avg_precision@10|   avg_recall@1000|        avg_ndcg@10|
+-------------------+------------------+-------------------+
|0.03359850257370144|0.6279566476339846|0.03210290644119731|
+-------------------+------------------+-------------------+



## LSH Recommender

In [8]:
lsh_rec = LSHRecommender()

lsh_rec, lsh_train_time = lsh_rec.fit(movies_profiles)
timing_results["LSH"] = {"train": lsh_train_time}
print(f"LSH Training Time: {lsh_train_time:.2f}s")

lsh_recs, lsh_inference_time = lsh_rec.recommend(test_user_profiles, movies_profiles, train_ratings)
timing_results["LSH"]["inference"] = lsh_inference_time
print(f"LSH Inference Time: {lsh_inference_time:.2f}s")

LSH Training Time: 0.09s


LSH Inference Time: 85.93s


In [9]:
lsh_metrics = evaluate_recommendations(lsh_recs, test_ratings)
print("LSH Metrics:")
lsh_metrics.show()

LSH Metrics:
+------------------+------------------+--------------------+
|  avg_precision@10|   avg_recall@1000|         avg_ndcg@10|
+------------------+------------------+--------------------+
|0.0129153018249883|0.3469084260173601|0.013980810053585948|
+------------------+------------------+--------------------+



## Hybrid Recommender

In [10]:
hybrid_rec = HybridRecommender(alpha=0.5)

timing_results["Hybrid"] = {"train": 0.0}
print("Hybrid Training Time: 0.00s (no training required)")

hybrid_recs, hybrid_inference_time = hybrid_rec.fuse(lsh_recs, als_recs)
timing_results["Hybrid"]["inference"] = hybrid_inference_time
print(f"Hybrid Inference Time: {hybrid_inference_time:.2f}s")

Hybrid Training Time: 0.00s (no training required)


Hybrid Inference Time: 199.51s


In [11]:
hybrid_metrics = evaluate_recommendations(hybrid_recs, test_ratings)
print("Hybrid Metrics:")
hybrid_metrics.show()

Hybrid Metrics:
+-------------------+--------------------+--------------------+
|   avg_precision@10|     avg_recall@1000|         avg_ndcg@10|
+-------------------+--------------------+--------------------+
|0.02721104351895178|0.017479625576318204|0.029816202018206134|
+-------------------+--------------------+--------------------+



## Timing Summary

In [12]:
import pandas as pd

timing_df = pd.DataFrame([
    {
        "Model": model,
        "Train (s)": times["train"],
        "Inference (s)": times["inference"],
        "Total (s)": times["train"] + times["inference"]
    }
    for model, times in timing_results.items()
])

timing_df

,Model,Train (s),Inference (s),Total (s)
0,ALS,6.085459,12.388300,18.473759
1,LSH,0.088112,85.931246,86.019358
2,Hybrid,0.000000,199.512028,199.512028


## Cleanup

In [13]:
spark.stop()